In [1]:
!pip install langchain langchain-google-genai langchain-community duckduckgo-search requests python-dotenv

In [2]:
import os
from getpass import getpass

# Set up API keys
os.environ["GOOGLE_API_KEY"] = getpass("Enter your Gemini API Key: ")
os.environ["WEATHER_API_KEY"] = getpass("Enter your Weather API Key: ")

Enter your Gemini API Key: ··········
Enter your Weather API Key: ··········


In [3]:
import requests
import json
from typing import Optional, Type
from langchain.tools import BaseTool, tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.tools import DuckDuckGoSearchRun
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

In [4]:
@tool
def get_weather_forecast(city: str) -> str:
    """
    Fetch current weather and 3-day forecast for a given city.

    Args:
        city: The name of the city to get weather information for

    Returns:
        Weather information including current conditions and 3-day forecast
    """
    api_key = os.environ.get("WEATHER_API_KEY")
    if not api_key:
        return "Weather API key not found. Please set WEATHER_API_KEY environment variable."

    base_url = "http://api.weatherapi.com/v1/forecast.json"
    params = {
        "key": api_key,
        "q": city,
        "days": 3,
        "aqi": "no",
        "alerts": "no"
    }

    try:
        response = requests.get(base_url, params=params)
        response.raise_for_status()
        data = response.json()

        # Extract relevant information
        current = data["current"]
        location = data["location"]
        forecast = data["forecast"]["forecastday"]

        weather_info = f"""
Weather Information for {location['name']}, {location['country']}:

Current Weather:
- Temperature: {current['temp_c']}°C ({current['temp_f']}°F)
- Condition: {current['condition']['text']}
- Humidity: {current['humidity']}%
- Wind: {current['wind_kph']} km/h
- Feels like: {current['feelslike_c']}°C

3-Day Forecast:
"""

        for day in forecast:
            date = day['date']
            day_data = day['day']
            weather_info += f"""
{date}:
- High: {day_data['maxtemp_c']}°C, Low: {day_data['mintemp_c']}°C
- Condition: {day_data['condition']['text']}
- Chance of rain: {day_data['daily_chance_of_rain']}%
"""

        return weather_info

    except requests.exceptions.RequestException as e:
        return f"Error fetching weather data: {str(e)}"
    except KeyError as e:
        return f"Error parsing weather data: {str(e)}"

In [7]:
@tool
def get_tourist_attractions(city: str) -> str:
    """
    Search for top tourist attractions and interesting places to visit in a given city.

    Args:
        city: The name of the city to search attractions for

    Returns:
        List of top tourist attractions and places to visit
    """
    try:
        # Initialize DuckDuckGo search
        search = DuckDuckGoSearchRun()

        # Search for tourist attractions
        query = f"top tourist attractions things to do visit {city} travel guide"
        search_results = search.run(query)

        return f"Top Tourist Attractions in {city}:\n\n{search_results}"

    except Exception as e:
        return f"Error searching for attractions: {str(e)}"

In [8]:
# Initialize Gemini LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",
    temperature=0.1,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)

# Create list of tools
tools = [get_weather_forecast, get_tourist_attractions]

# Create the prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an intelligent Travel Assistant AI. Your job is to help users plan their trips by providing:
    1. Current weather information and forecasts for their destination
    2. Top tourist attractions and interesting places to visit

    When a user asks about a destination, use the available tools to gather information and provide a comprehensive, helpful response.
    Always provide both weather information and tourist attractions unless specifically asked for only one.

    Be friendly, informative, and provide practical travel advice when relevant.

    Available tools:
    - get_weather_forecast: Get current weather and 3-day forecast for a city
    - get_tourist_attractions: Search for top tourist attractions in a city"""),
    ("user", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

# Create the tool calling agent
agent = create_tool_calling_agent(llm, tools, prompt)

# Create the agent executor
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [9]:
def travel_assistant(destination: str) -> str:
    """
    Main function to get travel information for a destination.

    Args:
        destination: The city/destination to get travel information for

    Returns:
        Comprehensive travel information including weather and attractions
    """
    try:
        # Create the input for the agent
        input_query = f"I'm planning to visit {destination}. Can you help me with the current weather forecast and top tourist attractions to visit there?"

        # Execute the agent
        response = agent_executor.invoke({"input": input_query})

        return response["output"]

    except Exception as e:
        return f"Error getting travel information: {str(e)}"

# Interactive function for user input
def interactive_travel_assistant():
    """
    Interactive function to continuously help users with travel planning.
    """
    print("🌍 Welcome to your Intelligent Travel Assistant! 🌍")
    print("I can help you with weather forecasts and tourist attractions for any destination.")
    print("Type 'quit' to exit.\n")

    while True:
        destination = input("Enter your destination city: ").strip()

        if destination.lower() in ['quit', 'exit', 'q']:
            print("Thank you for using the Travel Assistant! Safe travels! 🧳✈️")
            break

        if not destination:
            print("Please enter a valid destination.\n")
            continue

        print(f"\n🔍 Getting travel information for {destination}...\n")

        # Get travel information
        result = travel_assistant(destination)

        print("=" * 80)
        print(result)
        print("=" * 80)
        print()

In [10]:
# Test with a sample destination
print("Testing Travel Assistant with Paris...")
result = travel_assistant("Paris")
print(result)

Testing Travel Assistant with Paris...


> Entering new AgentExecutor chain...

Invoking: `get_weather_forecast` with `{'city': 'Paris'}`



Weather Information for Paris, France:

Current Weather:
- Temperature: 25.2°C (77.4°F)
- Condition: Partly Cloudy
- Humidity: 41%
- Wind: 12.2 km/h
- Feels like: 25.2°C

3-Day Forecast:

2025-06-16:
- High: 26.3°C, Low: 12.1°C
- Condition: Sunny
- Chance of rain: 0%

2025-06-17:
- High: 28.8°C, Low: 14.9°C
- Condition: Sunny
- Chance of rain: 0%

2025-06-18:
- High: 29.8°C, Low: 15.6°C
- Condition: Sunny
- Chance of rain: 0%

Invoking: `get_tourist_attractions` with `{'city': 'Paris'}`


Top Tourist Attractions in Paris:

Guide to the best hotels and things to do in Paris. Maps, travel tips and more. Our top recommendations for the best things to do in Paris, with pictures and travel tips from the editors at Conde Nast Traveler. Discover the 12 must-see attractions in Paris first-time visitors shouldn't miss—from the Eiffel Tower to Montmartre's 

In [11]:
# Run the interactive travel assistant
interactive_travel_assistant()

🌍 Welcome to your Intelligent Travel Assistant! 🌍
I can help you with weather forecasts and tourist attractions for any destination.
Type 'quit' to exit.

Enter your destination city: ooty

🔍 Getting travel information for ooty...



> Entering new AgentExecutor chain...

Invoking: `get_weather_forecast` with `{'city': 'ooty'}`



Weather Information for Ooty, India:

Current Weather:
- Temperature: 24.2°C (75.6°F)
- Condition: Mist
- Humidity: 89%
- Wind: 9.7 km/h
- Feels like: 26.7°C

3-Day Forecast:

2025-06-16:
- High: 20.1°C, Low: 16.9°C
- Condition: Patchy rain nearby
- Chance of rain: 88%

2025-06-17:
- High: 22.9°C, Low: 16.9°C
- Condition: Patchy rain nearby
- Chance of rain: 89%

2025-06-18:
- High: 24.8°C, Low: 16.5°C
- Condition: Patchy rain nearby
- Chance of rain: 82%

Invoking: `get_tourist_attractions` with `{'city': 'ooty'}`


Top Tourist Attractions in ooty:

Top 15 Best Things to Do in Ooty From tranquil boat rides to thrilling train journeys through lush hillsides, 